# 泰坦尼克号生存预测 —— 完整入门方案

按官方教程的思路，五步流程：

1. 加载数据
2. 探索性分析（EDA）
3. 特征工程
4. 训练模型 + 交叉验证
5. 预测测试集并生成提交文件 `submission.csv`

从第一个单元格开始，逐个 **Shift+Enter** 运行即可。


## 1. 加载数据


In [1]:
import sys
from pathlib import Path

# 让中文在 Windows 终端按 UTF-8 输出（避免 GBK 代码页下的乱码）
try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

# 数据与本 notebook 在同一目录。若当前工作目录不是 9-21-2026，
# 再尝试其下的 9-21-2026 子目录（取决于 VSCode 打开的工作区位置）。
BASE = Path.cwd()
if not (BASE / "train.csv").exists():
    BASE = BASE / "9-21-2026"

train = pd.read_csv(BASE / "train.csv")   # 训练集：有 Survived 标签
test = pd.read_csv(BASE / "test.csv")     # 测试集：没有标签，需要预测

print(f"训练集形状：{train.shape}（{train.shape[0]} 名乘客，{train.shape[1]} 个字段）")
print(f"测试集形状：{test.shape}（{test.shape[0]} 名乘客）")
print("\n字段含义：")
print("  Survived : 是否生还（0=否, 1=是）—— 我们要预测的目标")
print("  Pclass   : 舱位等级（1/2/3，代表社会经济地位）")
print("  Sex      : 性别")
print("  Age      : 年龄")
print("  SibSp    : 同行的兄弟姐妹/配偶数量")
print("  Parch    : 同行的父母/子女数量")
print("  Ticket   : 船票号")
print("  Fare     : 票价")
print("  Cabin    : 客舱号")
print("  Embarked : 登船港口（C=瑟堡, Q=皇后镇, S=南安普顿）")


训练集形状：(891, 12)（891 名乘客，12 个字段）
测试集形状：(418, 11)（418 名乘客）

字段含义：
  Survived : 是否生还（0=否, 1=是）—— 我们要预测的目标
  Pclass   : 舱位等级（1/2/3，代表社会经济地位）
  Sex      : 性别
  Age      : 年龄
  SibSp    : 同行的兄弟姐妹/配偶数量
  Parch    : 同行的父母/子女数量
  Ticket   : 船票号
  Fare     : 票价
  Cabin    : 客舱号
  Embarked : 登船港口（C=瑟堡, Q=皇后镇, S=南安普顿）


## 2. 探索性分析（EDA）


In [2]:
# 2.1 看看前几行数据长什么样
print("[2.1] 前 5 行数据：")
print(train.head())

# 2.2 缺失值情况 —— 缺失值需要在特征工程阶段处理
print("\n[2.2] 各字段缺失值数量：")
missing = train.isnull().sum()
missing = missing[missing > 0]
print(missing if len(missing) else "（无缺失值）")

# 2.3 目标变量分布 —— 生还率
print("\n[2.3] 目标变量 Survived 分布：")
print(train["Survived"].value_counts(normalize=True).round(3))
print(f"总体生还率：{train['Survived'].mean():.3f}（约 {train['Survived'].mean()*100:.1f}%）")


[2.1] 前 5 行数据：
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   Na

In [3]:
# 各特征与生存的关系 —— 这是特征工程的依据
print("[2.4] 性别与生还率（女性明显更高）：")
print(train.groupby("Sex")["Survived"].mean().round(3))

print("\n[2.5] 舱位等级与生还率（等级越高生还率越高）：")
print(train.groupby("Pclass")["Survived"].mean().round(3))

print("\n[2.6] 登船港口与生还率：")
print(train.groupby("Embarked")["Survived"].mean().round(3))

print("\n[2.7] 年龄段与生还率：")
print(train.groupby(pd.cut(train["Age"], bins=[0, 16, 30, 45, 60, 100]))["Survived"].mean().round(3))

print("\n[2.8] 同行亲属数量（SibSp）与生还率：")
print(train.groupby("SibSp")["Survived"].mean().round(3))


[2.4] 性别与生还率（女性明显更高）：
Sex
female    0.742
male      0.189
Name: Survived, dtype: float64

[2.5] 舱位等级与生还率（等级越高生还率越高）：
Pclass
1    0.630
2    0.473
3    0.242
Name: Survived, dtype: float64

[2.6] 登船港口与生还率：
Embarked
C    0.554
Q    0.390
S    0.337
Name: Survived, dtype: float64

[2.7] 年龄段与生还率：
Age
(0, 16]      0.550
(16, 30]     0.359
(30, 45]     0.426
(45, 60]     0.407
(60, 100]    0.227
Name: Survived, dtype: float64

[2.8] 同行亲属数量（SibSp）与生还率：
SibSp
0    0.345
1    0.536
2    0.464
3    0.250
4    0.167
5    0.000
8    0.000
Name: Survived, dtype: float64


## 3. 特征工程

结论：Sex、Pclass、Age 是最重要的三个特征，Embarked、SibSp/Parch 也有一定作用。下面把这些信息加工成模型能用的形式。


In [4]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """对原始数据做特征工程，返回加工后的 DataFrame（train 和 test 共用）。"""
    df = df.copy()

    # 3.1 从姓名中提取称呼（Title）：Mr/Mrs/Miss/Master 等，蕴含性别+年龄+身份信息
    df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
    # 把少见称呼合并成一类，避免过拟合
    df["Title"] = df["Title"].replace(
        ["Lady", "Countess", "Capt", "Col", "Don", "Dr",
         "Major", "Rev", "Sir", "Jonkheer", "Dona"], "Rare"
    )
    # 把等价称呼归并
    df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})

    # 3.2 家庭规模：自己 + 兄弟姐妹/配偶 + 父母/子女
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    # 是否独自一人（单独出行的人特征不同）
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    # 3.3 Cabin 缺失信息本身有意义：有没有客舱号记录
    df["HasCabin"] = df["Cabin"].notna().astype(int)

    # 3.4 缺失值填充
    # Age：按「舱位等级 + 性别」分组，用组内中位数填充（比全局中位数更精细）
    df["Age"] = df.groupby(["Pclass", "Sex"])["Age"].transform(lambda x: x.fillna(x.median()))
    # Embarked：用众数填充（只有极少数缺失）
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
    # Fare：用中位数填充（测试集有 1 个缺失）
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())

    # 3.5 把分类文字转换成数字
    df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
    df["Embarked"] = df["Embarked"].map({"S": 0, "C": 1, "Q": 2})
    df["Title"] = df["Title"].map({"Mr": 0, "Miss": 1, "Mrs": 2, "Master": 3, "Rare": 4})

    return df


In [5]:
train_fe = engineer_features(train)
test_fe = engineer_features(test)

# 模型用到的特征列
FEATURES = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare",
    "Embarked", "Title", "FamilySize", "IsAlone", "HasCabin",
]

X = train_fe[FEATURES]     # 特征矩阵
y = train_fe["Survived"]   # 标签

print(f"使用的特征（共 {len(FEATURES)} 个）：{FEATURES}")
print(f"特征矩阵形状：{X.shape}，标签形状：{y.shape}")
print(f"特征工程后缺失值数量：{X.isnull().sum().sum()}（应为 0）")


使用的特征（共 11 个）：['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone', 'HasCabin']
特征矩阵形状：(891, 11)，标签形状：(891,)
特征工程后缺失值数量：0（应为 0）


## 4. 训练模型 + 交叉验证


In [7]:
# 一个「朴素基准」：假设所有女性生还、所有男性遇难（官方示例 gender_submission 的思路）
baseline_pred = (train["Sex"] == "female").astype(int)
baseline_acc = accuracy_score(train["Survived"], baseline_pred)
print(f"[基准] 「女性全部生还」在训练集上的准确率：{baseline_acc:.4f}")

# 随机森林：对入门友好、鲁棒、不易过拟合、无需太多调参
model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)

# 5 折交叉验证：把训练集分成 5 份，轮流用 4 份训练、1 份验证，取平均，
# 这样评估的是「模型在没见过的数据上」的表现，而不是背下训练集。
scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print(f"[模型] 随机森林 5 折交叉验证准确率：{scores.mean():.4f}（±{scores.std():.4f}）")
print(f"       各折准确率：{np.round(scores, 4)}")
print(f"       相比基准提升：{(scores.mean() - baseline_acc) * 100:.2f} 个百分点")

model.fit(X, y)  # 用全部训练数据重新训练最终模型


[基准] 「女性全部生还」在训练集上的准确率：0.7868
[模型] 随机森林 5 折交叉验证准确率：0.8227（±0.0195）
       各折准确率：[0.838  0.8034 0.8258 0.7978 0.8483]
       相比基准提升：3.59 个百分点


UnicodeDecodeError: 'gbk' codec can't decode byte 0x94 in position 2952: illegal multibyte sequence

UnicodeDecodeError: 'gbk' codec can't decode byte 0x94 in position 2952: illegal multibyte sequence

RandomForestClassifier(max_depth=6, n_estimators=200, random_state=42)

## 5. 预测测试集并生成提交文件


In [8]:
X_test = test_fe[FEATURES]
predictions = model.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": predictions,
})
submission_path = BASE / "submission.csv"
submission.to_csv(submission_path, index=False)

print(f"已生成提交文件：{submission_path}")
print(f"共 {len(submission)} 条预测，其中预测生还 {predictions.sum()} 人")
print(f"预测生还率：{predictions.mean():.3f}")
print("\n提交文件前 10 行：")
print(submission.head(10))
print("\n完成！把 submission.csv 上传到 Kaggle 竞赛页面即可获得分数。")


已生成提交文件：c:\ml4biz\9-21-2026\submission.csv
共 418 条预测，其中预测生还 161 人
预测生还率：0.385

提交文件前 10 行：
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
5          897         0
6          898         1
7          899         0
8          900         1
9          901         0

完成！把 submission.csv 上传到 Kaggle 竞赛页面即可获得分数。
